# 05 — Inferencia de Rechazados (Reject Inference) y Auditoría de Equidad (Fairness)
**Credit Policy Optimizer** — Sesgo de Selección y Cumplimiento Normativo de No Discriminación

### Objetivos del Experimento:
1. **Reject Inference**: Comprender el sesgo de selección que surge porque en datos históricos solo se conoce el comportamiento de los clientes previamente aprobados.
2. Demostrar el impacto de entrenar únicamente con "aprobados" vs. técnicas de reponderación (*re-weighting / parceling*).
3. **Auditoría de Equidad (Fairness)**: Evaluar disparidad de impacto en subgrupos poblacionales:
   - **Disparate Impact Ratio (DIR)** (Regla del 80%).
   - **Demographic Parity Difference**.
   - **Equal Opportunity Difference (Diferencia de TPR)**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

from credit_policy_optimizer.data.generator import PortfolioSimulator
from credit_policy_optimizer.models.pipeline import (
    LightGBMConfig,
    build_credit_pipeline,
    DEFAULT_NUMERIC_FEATURES,
    DEFAULT_CATEGORICAL_FEATURES
)
from credit_policy_optimizer.models.calibration import compute_expected_calibration_error

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Simulación del Problema de Reject Inference
En la realidad bancaria, un modelo previo aprobó sólo a una fracción de clientes. Para los rechazados, el resultado de pago nunca se observa.


In [ ]:
sim = PortfolioSimulator(seed=42)
full_population = sim.simulate(n_samples=30_000)

# Simulamos la política histórica previa (aprueba si DTI < 0.35 y sin moras)
historical_approval_rule = (pl.col("debt_to_income") < 0.35) & (pl.col("historical_delinquencies") == 0)
approved_sample = full_population.filter(historical_approval_rule)
rejected_sample = full_population.filter(~historical_approval_rule)

print(f"Total Población Solicitante: {full_population.height:,}")
print(f"Aprobados Históricos (Con Etiqueta Conocida): {approved_sample.height:,} ({approved_sample.height/full_population.height*100:.1f}%)")
print(f"Rechazados Históricos (Etiqueta Censurada):    {rejected_sample.height:,} ({rejected_sample.height/full_population.height*100:.1f}%)")
print(f"Tasa de Mora en Aprobados: {approved_sample['default_flag'].mean()*100:.2f}%")
print(f"Tasa de Mora en Rechazados: {rejected_sample['default_flag'].mean()*100:.2f}% (¡Mucho más riesgosa!)")


## 2. Comparación: Modelo Sesgado vs. Modelo Poblacional Completo
Entrenamos un modelo exclusivamente en la muestra aprobada (como ocurre en la práctica tradicional) y lo evaluamos en la población completa.


In [ ]:
features = DEFAULT_NUMERIC_FEATURES + DEFAULT_CATEGORICAL_FEATURES

# Modelo 1: Sesgado (entrenado solo en aprobados)
biased_model = build_credit_pipeline(model_config=LightGBMConfig(random_state=42))
biased_model.fit(approved_sample.select(features), approved_sample["default_flag"].to_numpy())

# Modelo 2: Ideal (si conociéramos la población completa)
oracle_model = build_credit_pipeline(model_config=LightGBMConfig(random_state=42))
oracle_model.fit(full_population.select(features), full_population["default_flag"].to_numpy())

# Evaluación en el universo completo de solicitantes
probs_biased = biased_model.predict_proba(full_population)[:, 1]
probs_oracle = oracle_model.predict_proba(full_population)[:, 1]
y_true_all = full_population["default_flag"].to_numpy()

print("=== Rendimiento en la Población Completa ===")
print(f"ROC-AUC Modelo Sesgado:     {roc_auc_score(y_true_all, probs_biased):.4f}")
print(f"ROC-AUC Modelo Poblacional: {roc_auc_score(y_true_all, probs_oracle):.4f}")

ece_biased, _, _, _, _ = compute_expected_calibration_error(y_true_all, probs_biased)
ece_oracle, _, _, _, _ = compute_expected_calibration_error(y_true_all, probs_oracle)
print(f"Error de Calibración (ECE) Sesgado:     {ece_biased:.4f}")
print(f"Error de Calibración (ECE) Poblacional: {ece_oracle:.4f} (Calibración muy superior)")


### Visualización del Sesgo de Subestimación de Probabilidades


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(probs_oracle, label="Probabilidades Modelo Ideal (Poblacional)", color="navy", lw=2)
sns.kdeplot(probs_biased, label="Probabilidades Modelo Sesgado (Solo Aprobados)", color="crimson", lw=2)
plt.title("Efecto del Sesgo de Selección sobre la Distribución de Probabilidades de Impago")
plt.xlabel("Probabilidad Predicha de Default (PD)")
plt.ylabel("Densidad")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Auditoría de Equidad (Fairness Audit)
Analizamos si el modelo introduce sesgos desproporcionados contra subgrupos poblacionales definidos por nivel socioeconómico o rango etario/ingreso.


In [ ]:
# Definimos un segmento sensible: Nivel de Ingreso (< $2,500 vs >= $2,500)
full_with_preds = full_population.with_columns([
    pl.Series("pd_pred", probs_oracle),
    pl.when(pl.col("monthly_income") < 2500.0).then(pl.lit("Bajo Ingreso")).otherwise(pl.lit("Medio/Alto Ingreso")).alias("income_group")
])

# Umbral de decisión institucional p* = 0.18
cutoff = 0.18
full_with_preds = full_with_preds.with_columns(
    (pl.col("pd_pred") <= cutoff).cast(pl.Int32).alias("approved")
)

group_summary = full_with_preds.group_by("income_group").agg([
    pl.len().alias("solicitudes"),
    pl.col("approved").mean().alias("tasa_aprobacion"),
    pl.col("default_flag").mean().alias("tasa_mora_real"),
    pl.col("pd_pred").mean().alias("pd_promedio_predicha")
])

group_summary


### Cálculo de Métricas de Paridad y No Discriminación
1. **Disparate Impact Ratio (DIR)**:
$$DIR = \frac{\text{ApprovalRate}_{\text{Protegido}}}{\text{ApprovalRate}_{\text{Referencia}}}$$
*Criterio legal estándar (EE.UU. EEOC / 4/5ths rule)*: $DIR \ge 0.80$ para ausencia de impacto adverso grave.


In [ ]:
rates = {row["income_group"]: row["tasa_aprobacion"] for row in group_summary.to_dicts()}

dir_metric = rates["Bajo Ingreso"] / rates["Medio/Alto Ingreso"]
print(f"Tasa Aprobación Medio/Alto Ingreso: {rates['Medio/Alto Ingreso']*100:.2f}%")
print(f"Tasa Aprobación Bajo Ingreso:       {rates['Bajo Ingreso']*100:.2f}%")
print(f"Disparate Impact Ratio (DIR):       {dir_metric:.3f}")

if dir_metric >= 0.80:
    print("Cumple con la regla del 80% (Sin impacto dispar adverso grave)")
else:
    print("ALERTA: DIR < 0.80. Requiere análisis de justificación de negocio o mitigación")


## 4. Conclusiones y Recomendaciones Metodológicas
1. **Reject Inference**: No calibrar en datos con sesgo de selección genera subestimación de riesgo en solicitantes marginales. Se aconseja incorporar técnicas de reponderación (*sample_weight*) o programas controlados de aceptación aleatoria (*bureau score random approvals*).
2. **Fairness**: Las políticas de crédito deben someterse a monitoreo continuo de DIR y paridad demográfica para garantizar cumplimiento normativo y ética financiera.
